# Self-Play Drift Lab

Analyze recent self-play tournament performance for drift signals. Point the connection cell at the warehouse or application database replica before executing the queries.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from cerebro.core.config import settings

In [ ]:
DATABASE_URL = os.environ.get("CEREBRO_ANALYTICS_DB_URL", settings.database_url)
engine = create_engine(DATABASE_URL)

In [ ]:
window_days = int(os.environ.get("SELF_PLAY_WINDOW_DAYS", 14))
query = text(
    """
    SELECT
        scenario_id,
        started_at,
        ended_at,
        turns,
        tool_calls,
        success,
        fail_reason,
        metadata
    FROM self_play_matches
    WHERE started_at >= now() - (:window * interval '1 day')
    ORDER BY started_at
    """
)
matches = pd.read_sql_query(
    query.bindparams(window=window_days),
    engine,
    parse_dates=['started_at', 'ended_at'],
)
matches.head()

In [ ]:
if matches.empty:
    raise RuntimeError("No self-play matches found; adjust window or data source.")

matches['success_int'] = matches['success'].astype(int)
matches['started_at'] = pd.to_datetime(matches['started_at'], utc=True).dt.tz_convert(None)
scenario_groups = matches.groupby('scenario_id')
def _rolling_success(df):
    ordered = df.sort_values('started_at')
    series = (
        ordered
        .set_index('started_at')['success_int']
        .rolling('7D')
        .mean()
    )
    return series
rolling = scenario_groups.apply(_rolling_success)
rolling = rolling.rename('rolling_success').reset_index()
rolling.tail()

In [ ]:
turn_summary = (
    matches
    .groupby('scenario_id')
    .agg({
        'turns': ['mean', 'max'],
        'tool_calls': ['mean', 'max'],
        'success_int': 'mean'
    })
)
turn_summary.columns = ['turn_mean', 'turn_max', 'tool_calls_mean', 'tool_calls_max', 'success_rate']
turn_summary.sort_values('success_rate', ascending=True)

In [ ]:
scenario_filter = os.environ.get("SCENARIO_ID")
if scenario_filter:
    transcript_query = text(
        """
        SELECT
            started_at,
            transcript
        FROM self_play_matches
        WHERE scenario_id = :scenario_id
        ORDER BY started_at DESC
        LIMIT 5
        """
    )
    transcripts = pd.read_sql_query(
        transcript_query.bindparams(scenario_id=scenario_filter),
        engine,
        parse_dates=['started_at'],
    )
    transcripts['transcript'] = transcripts['transcript'].apply(lambda payload: json.dumps(payload, indent=2))
    transcripts